In [1]:
!uv pip install --upgrade pip
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision pandas

!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install tqdm wandb
!uv pip install bitsandbytes accelerate hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy==2.0"

Using Python 3.12.12 environment at: /usr
Resolved 1 package in 273ms
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
⠙ Preparing packages... (0/1)
Prepared 1 package in 158ms
Uninstalled 1 package in 196ms
Installed 1 package in 20ms
 - pip==24.1.2
 + pip==26.0.1
Using Python 3.12.12 environment at: /usr
Audited 3 packages in 280ms
Using Python 3.12.12 environment at: /usr
Uninstalled 2 packages in 754ms
 - pandas==2.3.3
 - torchvision==0.25.0+cu128
Using Python 3.12.12 en

In [2]:
import os
import re
import logging
import sys
import subprocess
import torch
import torch.nn as nn
import wandb
from typing import Dict, Tuple
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import os
import sys
from IPython import get_ipython
def configure_environment_paths():
    try:
        if "google.colab" in str(get_ipython()):
            print("✅ Environment: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            print("✅ Environment: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            print("⚠️ Environment: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        print("⚠️ Non-interactive session. Using local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"
    os.makedirs(base_output_path, exist_ok=True)
    print(f"📂 Data Path: {base_data_path}")
    print(f"📦 Output Path: {base_output_path}")
    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    env = ENV_NAME
    secret_value = None
    print(f"Attempting to load secret '{key_name}' from '{env}' environment...")
    try:
        if env == "colab":
            from google.colab import userdata

            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient

            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)
        if not secret_value:
            print(f"⚠️ Secret '{key_name}' not found in the {env} environment.")
            return None
        print(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as e:
        print(f"❌ An error occurred while loading secret '{key_name}': {e}")
        return None


def print_system_info():
    print("\n🔧 System Information")
    print(f"Python version: {sys.version.split()[0]}")
    try:
        import torch

        print(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            print(f"CUDA version: {torch.version.cuda}")
            print(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            print("CUDA not available")
    except ImportError:
        print("PyTorch not installed")
    finally:
        !nvidia-smi


INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle
print_system_info()

os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Now, these libraries will log in automatically
import wandb
import huggingface_hub

wandb.login()
huggingface_hub.login(token=os.environ["HF_TOKEN"])

2026-04-16 10:41:56.464383: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776336116.853122      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776336116.972032      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776336117.911691      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776336117.911728      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776336117.911731      23 computation_placer.cc:177] computation placer alr

✅ Environment: Kaggle
📂 Data Path: /kaggle/input/
📦 Output Path: /kaggle/working/

🔧 System Information
Python version: 3.12.12
PyTorch version: 2.10.0+cu128
CUDA version: 12.8
GPU count: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Thu Apr 16 10:42:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A 

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


✅ Successfully loaded secret 'GITHUB_TOKEN'.


wandb: Currently logged in as: dungngocpham171 (dungngocpham171-university-of-science) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
def download_from_hf(
    repo_id: str,
    local_dir: str = "ckpt",
    allow_patterns=None,
    force_download: bool = False,
    repo_type: str = "model",
):
    if allow_patterns is None:
        allow_patterns = ["*.safetensors", "scr*"]
    print(f"📥 Downloading checkpoint from Hugging Face Hub to '{local_dir}'...\n")
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        repo_type=repo_type,
        allow_patterns=allow_patterns,
        force_download=force_download,
    )
    print("\n✅ Download complete!")
    print(f"\n📂 Files in {local_dir}/:")
    for file in os.listdir(local_dir):
        if file.endswith(".safetensors"):
            size = os.path.getsize(os.path.join(local_dir, file)) / (1024**2)
            print(f"  ✓ {file} ({size:.2f} MB)")

In [4]:
!rm -rf ckpt
download_from_hf(
    repo_id="dzungpham/SLA-SemEval-challenge",
    local_dir="checkpoints",
    allow_patterns=["graphcodebert-robust/checkpoint-200*"],
)

📥 Downloading checkpoint from Hugging Face Hub to 'checkpoints'...



Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

graphcodebert-robust/checkpoint-200/scal(…):   0%|          | 0.00/1.38k [00:00<?, ?B/s]

graphcodebert-robust/checkpoint-200/mode(…):   0%|          | 0.00/499M [00:00<?, ?B/s]

graphcodebert-robust/checkpoint-200/opti(…):   0%|          | 0.00/4.74M [00:00<?, ?B/s]

...

trainer_state.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

graphcodebert-robust/checkpoint-200/trai(…):   0%|          | 0.00/5.84k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]


✅ Download complete!

📂 Files in checkpoints/:


In [5]:
# ============================================================================
# 1. DATA LAYER: Load and preprocess datasets
# ============================================================================

import random
import os
import re
import logging
import numpy as np
from typing import Dict, Tuple
from datasets import load_dataset, Dataset, concatenate_datasets

# Define a logger for this module to avoid NameError
logger = logging.getLogger(__name__)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

def preprocess_code(code_str: str) -> str:
    """Normalize code string with semantic-preserving perturbations for training."""
    code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
    code_str = re.sub(r"\r\n|\r", "\n", code_str)

    # 20% chance to replace 4 spaces with tabs
    if random.random() < 0.2:
        code_str = code_str.replace("    ", "\t")
    # 20% chance to replace tabs with 2 spaces
    elif random.random() < 0.2:
        code_str = code_str.replace("    ", "  ")

    code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
    code_str = re.sub(r"\n{3,}", "\n\n", code_str)
    code_str = re.sub(r"[ \t]+", " ", code_str)
    return code_str.strip()


def tokenize_function(examples: Dict, tokenizer, max_length: int = 512) -> Dict:
    """Tokenize code examples."""
    codes = [preprocess_code(c) for c in examples["code"]]
    return tokenizer(codes, truncation=True, max_length=max_length, padding=False)


def upsample_dataset(dataset: Dataset) -> Dataset:
    """Upsample the minority class to match the majority class size."""
    labels = np.array(dataset["label"])
    class_counts = np.bincount(labels)
    majority_class = np.argmax(class_counts)
    minority_class = np.argmin(class_counts)

    logger.info(f"Initial distribution: Class 0: {class_counts[0]}, Class 1: {class_counts[1]}")

    diff = class_counts[majority_class] - class_counts[minority_class]

    if diff > 0:
        minority_indices = np.where(labels == minority_class)[0]
        upsample_indices = np.random.choice(minority_indices, size=diff, replace=True)
        upsampled_data = dataset.select(upsample_indices)
        dataset = concatenate_datasets([dataset, upsampled_data])

        new_counts = np.bincount(np.array(dataset["label"]))
        logger.info(f"Balanced distribution: Class 0: {new_counts[0]}, Class 1: {new_counts[1]}")

    return dataset.shuffle(seed=42)


def load_datasets(tokenizer, max_length: int = 512, aug=None) -> Tuple[Dataset, Dataset]:
    """Load and tokenize train and validation datasets with upsampling."""
    logger.info("Loading datasets from Hugging Face Hub...")

    train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
    val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

    logger.info(f"Train samples (before upsampling): {len(train_dataset)}")
    # train_dataset = upsample_dataset(train_dataset)

    logger.info(f"Final samples: Train: {len(train_dataset)}, Val: {len(val_dataset)}")

    if aug is not None:
        logger.info("Applying data augmentation before tokenization...")
        def augment_batch(examples):
            examples["code"] = [aug(c) for c in examples["code"]]
            return examples
        train_dataset = train_dataset.map(
            augment_batch,
            batched=True,
            batch_size=512,
            desc="Augmenting train",
            num_proc=os.cpu_count(),
        )

    # Tokenize
    logger.info("Tokenizing datasets...")
    train_dataset = train_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in train_dataset.column_names],
        desc="Tokenizing train",
        num_proc=os.cpu_count(),
    )

    val_dataset = val_dataset.map(
        lambda ex: tokenize_function(ex, tokenizer, max_length),
        batched=True,
        batch_size=512,
        remove_columns=[col for col in ["code", "language"] if col in val_dataset.column_names],
        desc="Tokenizing val",
        num_proc=os.cpu_count(),
    )

    # Rename label column
    train_dataset = train_dataset.rename_column("label", "labels")
    val_dataset = val_dataset.rename_column("label", "labels")

    return train_dataset, val_dataset

# ============================================================================
# 1.5 METRICS: Evaluation metric function for Trainer
# ============================================================================

def compute_metrics_fn(eval_pred):
    """Compute precision, recall, F1, and accuracy for evaluation."""
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)

    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

In [6]:
# ============================================================================
# 3. TRAINING ENGINE: HF Trainer handles training, validation, and logging
# ============================================================================

# Note: The Trainer class from transformers handles:
# - Training loop with gradient accumulation
# - Validation every N steps (eval_steps=200)
# - Best model selection based on macro_f1
# - Automatic W&B logging if enabled
# - Checkpointing and model saving

In [7]:
# ============================================================================
# 2. MODEL LAYER: Load pretrained CodeBERT and tokenizer
# ============================================================================

def load_model_and_tokenizer(model_name: str, num_labels: int = 2):
    """Load pretrained model and tokenizer from Hugging Face."""
    logger.info(f"Loading tokenizer from: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    logger.info(f"Loading model from: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        problem_type="single_label_classification",
    )

    logger.info(f"Model loaded successfully")
    logger.info(f"Model config - num_labels: {model.config.num_labels}, hidden_size: {model.config.hidden_size}")

    return model, tokenizer


def freeze_base_model(model):
    """
    Freeze all base model parameters, keeping only classifier head trainable.

    For sequence classification models (e.g., CodeBERT), this freezes the
    RoBERTa/encoder layers and unfreezes the classification head.

    Args:
        model: The pretrained model to freeze
    """
    # Freeze all encoder/base model parameters
    for name, param in model.named_parameters():
        if "classifier" not in name and "cls" not in name.lower():
            param.requires_grad = False

    # Ensure classifier head is trainable
    for name, param in model.named_parameters():
        if "classifier" in name or "cls" in name.lower():
            param.requires_grad = True

    logger.info("Base model frozen - only classifier head is trainable")


In [8]:
# =============================================================================
# 3. TRAINING ENGINE: HF Trainer with enhanced regularization for OOD generalization
# =============================================================================

import os
import logging
import random
import numpy as np
from dataclasses import dataclass, field
from typing import Callable, Optional, Tuple, List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    PreTrainedTokenizer,
)
from transformers.modeling_outputs import SequenceClassifierOutput

# Disable tokenizers parallelism warnings if desired
os.environ["TOKENIZERS_PARALLELISM"] = "true"

# ----------------------------------------------------------------------
# Configuration dataclass (extended)
# ----------------------------------------------------------------------
@dataclass
class TrainConfig:
    model_name: str = "microsoft/graphcodebert-base"
    output_dir: str = "./graphcodebert"
    num_epochs: int = 3
    batch_size: int = 16
    learning_rate: float = 2e-5
    max_length: int = 512
    num_labels: int = 2
    use_wandb: bool = False
    freeze_base: bool = False
    loss_type: str = "r-drop"           # "ce", "focal", "infonce", "r-drop"
    focal_alpha: float = 1.0
    focal_gamma: float = 2.0
    r_drop_alpha: float = 4.0
    infonce_temperature: float = 0.07
    infonce_weight: float = 0.5
    seed: int = 42
    resume_from_checkpoint: str = None

    # ----- new regularisation parameters -----
    label_smoothing: float = 0.1            # label smoothing factor (0 = no smoothing)
    adversarial_epsilon: float = 0.5        # FGM perturbation magnitude (0 = disable)
    use_swa: bool = True                    # enable Stochastic Weight Averaging
    swa_start_epoch: int = 2                # start SWA after this epoch
    swa_lr: float = 1e-5                    # learning rate for SWA (often lower)
    data_augmentation: bool = True          # enable code augmentation
    aug_rename_prob: float = 0.3            # probability to rename identifiers
    aug_format_prob: float = 0.3            # probability to add/remove blank lines

    # Internally populated
    device: torch.device = field(init=False, default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu"))


# ----------------------------------------------------------------------
# Logging utilities
# ----------------------------------------------------------------------
def setup_logger(output_dir: str) -> logging.Logger:
    """Create a logger that writes to both console and a file."""
    logger = logging.getLogger("train_pipeline")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if logger.hasHandlers():
        logger.handlers.clear()

    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    ch.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
    logger.addHandler(ch)

    log_path = os.path.join(output_dir, "training.log")
    fh = logging.FileHandler(log_path, mode="w")
    fh.setLevel(logging.INFO)
    fh.setFormatter(
        logging.Formatter("%(asctime)s - %(levelname)s - %(name)s - %(message)s")
    )
    logger.addHandler(fh)

    logger.info(f"Logging to {log_path}")
    return logger


# ----------------------------------------------------------------------
# Model / tokenizer loading
# ----------------------------------------------------------------------
def load_model_and_tokenizer(
    model_name: str,
    num_labels: int,
    device: torch.device,
    logger: logging.Logger,
) -> Tuple[nn.Module, PreTrainedTokenizer]:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    logger.info(f"Loading model & tokenizer for '{model_name}'")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
        hidden_dropout_prob=0.2,
        attention_probs_dropout_prob=0.2,
    )
    model.to(device)
    logger.info(f"Model placed on {device}")
    return model, tokenizer


def freeze_base_model(model: nn.Module, logger: logging.Logger) -> None:
    """Freeze all parameters except the classification head."""
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False
    logger.info("Base model weights frozen – only classifier head will be trained.")


# ----------------------------------------------------------------------
# Loss function factories (with label smoothing support)
# ----------------------------------------------------------------------
def get_label_smoothed_cross_entropy(smoothing: float) -> Callable:
    """Return a loss function that applies label smoothing."""
    def loss_fn(outputs, labels, **_):
        logits = outputs.logits
        n_classes = logits.size(-1)
        # Create smoothed targets
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
        return -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()
    return loss_fn


def get_focal_loss(alpha: float, gamma: float, smoothing: float = 0.0) -> Callable:
    def focal_loss(outputs, labels, **_):
        logits = outputs.logits
        if smoothing > 0:
            # Simplified focal with label smoothing: apply smoothing to one-hot
            n_classes = logits.size(-1)
            smooth_targets = torch.zeros_like(logits).fill_(smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
            ce = -(smooth_targets * F.log_softmax(logits, dim=-1)).sum(dim=-1)
        else:
            ce = F.cross_entropy(logits, labels, reduction="none")
        pt = torch.exp(-ce)
        loss = alpha * (1 - pt) ** gamma * ce
        return loss.mean()
    return focal_loss


def get_infonce_loss(temperature: float, weight: float, smoothing: float = 0.0) -> Callable:
    def infonce_loss(outputs, labels, **_):
        reps = outputs.hidden_states[-1]
        reps = F.normalize(reps, dim=-1)
        sim_matrix = torch.mm(reps, reps.t()) / temperature
        target = torch.arange(reps.size(0), device=reps.device)
        # Ignore label smoothing for InfoNCE (contrastive)
        loss = F.cross_entropy(sim_matrix, target)
        return weight * loss
    return infonce_loss


# ----------------------------------------------------------------------
# Helper: log model architecture
# ----------------------------------------------------------------------
def log_model_architecture(model: nn.Module, tokenizer, logger: logging.Logger) -> None:
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params

    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")

    logger.info("===== Tokenizer Summary =====")
    logger.info(f"Vocab size: {len(tokenizer)} | Special tokens: {tokenizer.all_special_tokens}")
    logger.info("===== End of Architecture Log =====")


# ----------------------------------------------------------------------
# Data augmentation for code (simple but effective)
# ----------------------------------------------------------------------
class CodeAugmentation:
    """Apply lightweight code transformations to improve robustness."""
    def __init__(self, rename_prob: float = 0.3, format_prob: float = 0.3):
        self.rename_prob = rename_prob
        self.format_prob = format_prob

    def __call__(self, text: str) -> str:
        # Clone input string
        text = str(text)
        # 1. Identifier renaming (replace variable/function names with random tokens)
        if random.random() < self.rename_prob:
            # Very simple: replace all alphabetic tokens that are not keywords
            import re
            tokens = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', text)
            # filter out common Python keywords (simplified)
            keywords = {'def', 'return', 'if', 'else', 'for', 'while', 'import', 'from', 'as', 'with', 'try', 'except', 'class', 'lambda', 'and', 'or', 'not', 'True', 'False', 'None'}
            identifiers = [t for t in set(tokens) if t not in keywords and len(t) > 1]
            if identifiers:
                rename_map = {id: f"var_{random.randint(1000,9999)}" for id in identifiers[:5]}
                for old, new in rename_map.items():
                    text = text.replace(old, new)

        # 2. Formatting changes: add/remove blank lines and spaces
        if random.random() < self.format_prob:
            lines = text.split('\n')
            new_lines = []
            for line in lines:
                new_lines.append(line)
                # randomly insert blank line
                if random.random() < 0.1:
                    new_lines.append('')
            text = '\n'.join(new_lines)
            # also add random spaces at start of lines
            if random.random() < 0.2:
                text = '\n'.join(' ' + line if random.random() < 0.1 else line for line in text.split('\n'))

        return text


class AugmentedDataset(torch.utils.data.Dataset):
    """Wrap a dataset and apply code augmentation on the fly."""
    def __init__(self, dataset, tokenizer, max_length: int, aug: CodeAugmentation):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.aug = aug

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Assume the dataset has 'code' field (or 'text') – adapt as needed
        if 'code' in item:
            code = item['code']
        elif 'text' in item:
            code = item['text']
        else:
            raise KeyError("Dataset must contain 'code' or 'text' key for augmentation")
        # Apply augmentation to code
        code = self.aug(code)
        # Tokenize
        encoding = self.tokenizer(
            code,
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors=None
        )
        # Keep original label
        label = item['label']
        return {**encoding, 'labels': label}


# ----------------------------------------------------------------------
# Custom Trainer with label smoothing, adversarial training, and SWA
# ----------------------------------------------------------------------
class RobustTrainer(Trainer):
    """Trainer supporting label smoothing, FGM adversarial training, and SWA."""
    def __init__(
        self,
        *args,
        loss_type: str = "ce",
        r_drop_alpha: float = 4.0,
        compute_loss_fn: Optional[Callable] = None,
        label_smoothing: float = 0.0,
        adversarial_epsilon: float = 0.0,
        use_swa: bool = False,
        swa_start_epoch: int = 2,
        swa_lr: float = 1e-5,
        **kwargs
    ):
        super().__init__(*args, **kwargs)
        self.loss_type = loss_type
        self.r_drop_alpha = r_drop_alpha
        self.compute_loss_fn = compute_loss_fn
        self.label_smoothing = label_smoothing
        self.adversarial_epsilon = adversarial_epsilon
        self.use_swa = use_swa
        self.swa_start_epoch = swa_start_epoch
        self.swa_lr = swa_lr
        self.swa_model = None
        self.swa_scheduler = None
        self._swa_started = False

    def compute_loss(self, model, inputs, num_items_in_batch = 64, return_outputs: bool = False):
        """Override to incorporate label smoothing and adversarial training."""
        labels = inputs.pop("labels", None)

        # ---- R-Drop (dual forward) ----
        if self.loss_type == "r-drop" and model.training and labels is not None:
            out1 = model(**inputs)
            out2 = model(**inputs)
            ce1 = F.cross_entropy(out1.logits, labels)
            ce2 = F.cross_entropy(out2.logits, labels)
            ce = (ce1 + ce2) / 2

            p = F.log_softmax(out1.logits, dim=-1)
            q = F.log_softmax(out2.logits, dim=-1)
            kl = (F.kl_div(p, F.softmax(out2.logits, dim=-1), reduction="batchmean") +
                  F.kl_div(q, F.softmax(out1.logits, dim=-1), reduction="batchmean")) / 2

            loss = ce + self.r_drop_alpha * kl
            return (loss, out1) if return_outputs else loss

        # ---- Standard forward (potentially with adversarial training) ----
        # First clean forward
        outputs = model(**inputs)
        if labels is None:
            loss = outputs.loss if hasattr(outputs, "loss") else outputs[0]
        else:
            if self.compute_loss_fn:
                loss = self.compute_loss_fn(outputs, labels)
            else:
                # Use label smoothing if enabled and not overridden
                if self.label_smoothing > 0 and self.loss_type == "ce":
                    loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                    loss = loss_fn(outputs, labels)
                else:
                    loss = F.cross_entropy(outputs.logits, labels)

        # ---- Adversarial training (FGM) ----
        if self.adversarial_epsilon > 0 and model.training and labels is not None:
            # Backward to compute gradients for perturbation
            loss.backward(retain_graph=True)
            # Store original parameters and perturb
            embeddings = model.roberta.embeddings.word_embeddings.weight
            grad = embeddings.grad
            if grad is not None:
                perturbation = self.adversarial_epsilon * grad.sign()
                embeddings.data.add_(perturbation)

                # Forward with perturbed embeddings
                adv_outputs = model(**inputs)
                if self.compute_loss_fn:
                    adv_loss = self.compute_loss_fn(adv_outputs, labels)
                else:
                    if self.label_smoothing > 0 and self.loss_type == "ce":
                        loss_fn = get_label_smoothed_cross_entropy(self.label_smoothing)
                        adv_loss = loss_fn(adv_outputs, labels)
                    else:
                        adv_loss = F.cross_entropy(adv_outputs.logits, labels)

                # Combine losses (original already computed, we can sum)
                loss = loss + adv_loss

                # Restore embeddings
                embeddings.data.sub_(perturbation)
                model.zero_grad()

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch):
        """Override to handle SWA model updates and optionally reset optimizer."""
        loss = super().training_step(model, inputs, num_items_in_batch)

        # Initialize SWA if we reached the start epoch and not yet started
        if self.use_swa and not self._swa_started and self.state.epoch >= self.swa_start_epoch:
            self._swa_started = True
            self.swa_model = AveragedModel(model)
            self.swa_scheduler = SWALR(self.optimizer, swa_lr=self.swa_lr)
            self.log({"SWA": "started"})

        # Update SWA moving average after each step
        if self._swa_started and self.swa_model is not None:
            self.swa_model.update_parameters(model)

        return loss

    def save_model(self, output_dir: Optional[str] = None, _internal_call: bool = False):
        """Override to save SWA model if enabled."""
        if self._swa_started and self.swa_model is not None:
            # Update batch norm statistics before saving
            if self.train_dataset is not None:
                update_bn(self.get_train_dataloader(), self.swa_model, device=self.args.device)  # ← fixed
            # Save SWA model
            model_to_save = self.swa_model.module if hasattr(self.swa_model, 'module') else self.swa_model
            model_to_save.save_pretrained(output_dir or self.args.output_dir)
        else:
            super().save_model(output_dir, _internal_call)

# ----------------------------------------------------------------------
# Metrics function
# ----------------------------------------------------------------------
def compute_metrics_fn(eval_pred):
    """Compute macro F1 and accuracy."""
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    from sklearn.metrics import f1_score, accuracy_score
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }


# ----------------------------------------------------------------------
# Main training pipeline
# ----------------------------------------------------------------------
def train_pipeline(cfg: TrainConfig) -> Tuple[Trainer, nn.Module, PreTrainedTokenizer]:
    os.makedirs(cfg.output_dir, exist_ok=True)
    logger = setup_logger(cfg.output_dir)

    # Log configuration
    logger.info(f"Training config: {cfg}")

    # Load model & tokenizer
    model, tokenizer = load_model_and_tokenizer(
        cfg.model_name, cfg.num_labels, cfg.device, logger
    )

    # Freeze base if requested
    if cfg.freeze_base:
        freeze_base_model(model, logger)

    # Log architecture
    log_model_architecture(model, tokenizer, logger)

    # Enable hidden states for InfoNCE
    if cfg.loss_type == "infonce":
        model.config.output_hidden_states = True
        logger.info("Enabled hidden states for InfoNCE loss.")

    # Set up optional augmentation
    aug = None
    if cfg.data_augmentation:
        aug = CodeAugmentation(rename_prob=cfg.aug_rename_prob, format_prob=cfg.aug_format_prob)
        logger.info(f"Data augmentation enabled (rename={cfg.aug_rename_prob}, format={cfg.aug_format_prob})")

    # Load datasets with optional static augmentation before tokenization
    train_dataset, val_dataset = load_datasets(tokenizer, cfg.max_length, aug=aug)

    # WandB optional
    if cfg.use_wandb:
        try:
            import wandb
            os.environ["WANDB_MODE"] = "online"
        except Exception as e:
            logger.warning(f"W&B import failed ({e}); proceeding without it.")
            cfg.use_wandb = False

    # Steps calculation
    steps_per_epoch = max(1, len(train_dataset) // cfg.batch_size)
    total_steps = cfg.num_epochs * steps_per_epoch
    warmup_steps = max(1, int(total_steps * 0.1))

    # Training arguments
    training_args = TrainingArguments(
        output_dir=cfg.output_dir,
        num_train_epochs=cfg.num_epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size * 2,
        learning_rate=cfg.learning_rate,
        warmup_steps=warmup_steps,
        weight_decay=0.1,
        logging_strategy="steps",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=1000,
        save_strategy="steps",
        save_steps=200,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=4,
        report_to=["wandb"] if cfg.use_wandb else [],
        run_name="graphcodebert-robust" if cfg.use_wandb else None,
        dataloader_pin_memory=torch.cuda.is_available(),
        seed=cfg.seed,
        fp16=True,
    )

    data_collator = DataCollatorWithPadding(tokenizer)

    # Loss function selection (use label smoothing inside compute_loss if not overridden)
    compute_loss_fn = None
    if cfg.loss_type == "focal":
        compute_loss_fn = get_focal_loss(cfg.focal_alpha, cfg.focal_gamma, smoothing=cfg.label_smoothing if cfg.loss_type=="ce" else 0.0)
    elif cfg.loss_type == "infonce":
        compute_loss_fn = get_infonce_loss(cfg.infonce_temperature, cfg.infonce_weight)

    # Instantiate robust trainer
    trainer = RobustTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics_fn,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        loss_type=cfg.loss_type,
        r_drop_alpha=cfg.r_drop_alpha,
        compute_loss_fn=compute_loss_fn,
        label_smoothing=cfg.label_smoothing,
        adversarial_epsilon=cfg.adversarial_epsilon,
        use_swa=cfg.use_swa,
        swa_start_epoch=cfg.swa_start_epoch,
        swa_lr=cfg.swa_lr,
    )

    logger.info("=== Starting training with robust regularisation ===")
    trainer.train(resume_from_checkpoint=cfg.resume_from_checkpoint)
    logger.info("Training complete!")

    # Final model (SWA averaged if used)
    final_model = trainer.swa_model if (trainer._swa_started and trainer.swa_model) else model
    final_model.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    tokenizer.save_pretrained(os.path.join(cfg.output_dir, "final_model"))
    logger.info(f"Best model saved to {os.path.join(cfg.output_dir, 'best_model')}")
    logger.info(f"Final model (SWA averaged) saved to {os.path.join(cfg.output_dir, 'final_model')}")

    return trainer, final_model, tokenizer


# ----------------------------------------------------------------------
# Cleanup and example usage
# ----------------------------------------------------------------------
# if __name__ == "__main__":
#     import gc
#     gc.collect()
#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

#     cfg = TrainConfig(
#         model_name="microsoft/graphcodebert-base",
#         output_dir="./output_checkpoints/graphcodebert-robust",
#         num_epochs=5,
#         batch_size=32,
#         learning_rate=2e-5,
#         max_length=512,
#         use_wandb=True,
#         freeze_base=True,
#         loss_type="r-drop",
#         r_drop_alpha=4.0,
#         label_smoothing=0.1,
#         adversarial_epsilon=0.5,
#         use_swa=True,
#         swa_start_epoch=2,
#         swa_lr=1e-5,
#         data_augmentation=True,
#         aug_rename_prob=0.3,
#         aug_format_prob=0.3,
#         resume_from_checkpoint="output_checkpoints/graphcodebert-robust/checkpoint-1000",
#     )
#     trainer, model, tokenizer = train_pipeline(cfg)

In [9]:
import os
import logging
from pathlib import Path
from typing import Tuple, Dict, Any

import numpy as np
import torch
from tqdm.auto import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
# New imports for metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ----------------------------------------------------------------------
# 1. SETUP LOGGER
# ----------------------------------------------------------------------
def setup_logger(output_dir: str) -> logging.Logger:
    logger = logging.getLogger("inference")
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        ch = logging.StreamHandler()
        ch.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
        logger.addHandler(ch)

        os.makedirs(output_dir, exist_ok=True)
        log_path = Path(output_dir) / "inference.log"
        fh = logging.FileHandler(log_path, mode="w")
        fh.setFormatter(logging.Formatter("%(asctime)s - %(levelname)s - %(message)s"))
        logger.addHandler(fh)
    return logger

def log_model_architecture(model: nn.Module, tokenizer, logger: logging.Logger) -> None:
    """
    Print a concise but complete description of the model and tokenizer.
    Also calculates and displays trainable vs non-trainable parameters.
    """
    logger.info("===== Model Architecture =====")
    logger.info("\n" + model.__repr__())

    # Parameter Counting
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    total_params = trainable_params + non_trainable_params

    logger.info("===== Parameter Summary =====")
    logger.info(f"Total Parameters:         {total_params:,}")
    logger.info(f"Trainable Parameters:     {trainable_params:,}")
    logger.info(f"Non-trainable Parameters: {non_trainable_params:,}")

    # Tokenizer summary
    logger.info("===== Tokenizer Summary =====")
    logger.info(
        f"Vocab size: {len(tokenizer)} | "
        f"Special tokens: {tokenizer.all_special_tokens}"
    )
    logger.info("===== End of Architecture Log =====")

# ----------------------------------------------------------------------
# 2. MAIN INFERENCE PIPELINE WITH METRICS
# ----------------------------------------------------------------------
def run_standalone_inference(
    checkpoint_path: str,
    output_dir: str = "./",
    output_csv: str = "submission.csv",
    batch_size: int = 32,
    max_length: int = 512,
    dataset_name: str = "dzungpham/SemEval-2026-TaskA-dataset",
    dataset_config: str = "default",
    split: str = "test",
):
    logger = setup_logger(output_dir)
    logger.info(f"Loading model and tokenizer from: {checkpoint_path}")

    # --- Load Model & Tokenizer ---
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
    log_model_architecture(model, tokenizer, logger)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # --- Load Dataset ---
    logger.info(f"Loading dataset: {dataset_name} ({dataset_config})")
    try:
        test_ds = load_dataset(dataset_name, dataset_config, split=split)
    except Exception as e:
        logger.warning(f"Default loading failed due to schema mismatch: {e}")
        logger.info(f"Attempting to load split '{split}' using data_files...")
        # Fallback: load only the test split files to avoid schema conflict with train/val
        test_ds = load_dataset(dataset_name, data_files={split: f"*{split}*"}, split=split)

    # --- Tokenization ---
    def tokenize_fn(examples):
        return tokenizer(
            examples["code"],
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )

    logger.info("Tokenizing dataset...")
    tokenized_ds = test_ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=[c for c in test_ds.column_names if c not in ["input_ids", "attention_mask", "id", "label"]],
        desc="Tokenizing"
    )
    tokenized_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

    # --- DataLoader ---
    test_loader = torch.utils.data.DataLoader(
        tokenized_ds,
        batch_size=batch_size,
        shuffle=False,
    )

    # --- Inference Loop ---
    logger.info(f"Running inference on {len(test_ds)} examples...")
    all_logits = []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            inputs = {k: v.to(device) for k, v in batch.items() if k in ["input_ids", "attention_mask"]}
            outputs = model(**inputs)
            all_logits.append(outputs.logits.cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    pred_labels = logits.argmax(axis=-1)

    # ------------------------------------------------------------------
    # 3. METRIC CALCULATION
    # ------------------------------------------------------------------
    if "label" in test_ds.column_names:
        logger.info("Calculating classification metrics...")
        true_labels = np.array(test_ds["label"])

        # Calculate Accuracy
        acc = accuracy_score(true_labels, pred_labels)

        # Calculate Precision, Recall, F1 (weighted handles class imbalance)
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, pred_labels, average='weighted'
        )

        logger.info("-" * 30)
        logger.info(f"METRICS FOR SPLIT: {split}")
        logger.info(f"Accuracy:  {acc:.4f}")
        logger.info(f"Precision: {precision:.4f}")
        logger.info(f"Recall:    {recall:.4f}")
        logger.info(f"F1-Score:  {f1:.4f}")
        logger.info("-" * 30)

        # Optional: Print Confusion Matrix
        cm = confusion_matrix(true_labels, pred_labels)
        logger.info(f"Confusion Matrix:\n{cm}")
    else:
        logger.warning("No 'label' column found in dataset. Skipping metric calculation.")

    # --- Save Results ---
    csv_path = Path(output_dir) / output_csv

    if "id" in test_ds.column_names:
        ids = test_ds["id"]
    elif "ID" in test_ds.column_names:
        ids = test_ds["ID"]
    else:
        ids = list(range(len(pred_labels)))

    with open(csv_path, "w", encoding="utf-8") as f:
        f.write("id,label\n")
        for idx, label in zip(ids, pred_labels):
            f.write(f"{idx},{label}\n")

    logger.info(f"✅ Predictions saved to {csv_path}")

# ----------------------------------------------------------------------
# 4. EXECUTION
# ----------------------------------------------------------------------
if __name__ == "__main__":
    CHECKPOINT = "checkpoints/graphcodebert-robust/checkpoint-200"
    OUTPUT_DIR = "test/inference/graphcodebert-robust"
    !mkdir -p OUTPUT_DIR
    if os.path.exists(CHECKPOINT):
        run_standalone_inference(
            checkpoint_path=CHECKPOINT,
            output_dir=OUTPUT_DIR,
            output_csv="submission.csv",
            batch_size=64,
            dataset_name="dzungpham/SemEval-2026-TaskA-dataset",
            dataset_config="default",
            split="test"
        )


2026-04-16 10:42:50,167 - INFO - Loading model and tokenizer from: checkpoints/graphcodebert-robust/checkpoint-200
INFO:inference:Loading model and tokenizer from: checkpoints/graphcodebert-robust/checkpoint-200
2026-04-16 10:42:50,469 - INFO - ===== Model Architecture =====
INFO:inference:===== Model Architecture =====
2026-04-16 10:42:50,471 - INFO - 
RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, 

train.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/298M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/500000 [00:00<?, ? examples/s]

...

2026-04-16 10:43:21,380 - INFO - Tokenizing dataset...
INFO:inference:Tokenizing dataset...


Tokenizing:   0%|          | 0/500000 [00:00<?, ? examples/s]

2026-04-16 10:48:34,156 - INFO - Running inference on 500000 examples...
INFO:inference:Running inference on 500000 examples...


Predicting:   0%|          | 0/7813 [00:00<?, ?it/s]

2026-04-16 15:41:53,221 - WARNING - No 'label' column found in dataset. Skipping metric calculation.
2026-04-16 15:41:59,383 - INFO - ✅ Predictions saved to test/inference/graphcodebert-robust/submission.csv
INFO:inference:✅ Predictions saved to test/inference/graphcodebert-robust/submission.csv


In [10]:
from huggingface_hub import HfApi

api = HfApi()

# Replace these with your details
repo_id = "dzungpham/SLA-SemEval-challenge"
local_folder_path = "test"

# Create the repo first (optional, will skip if it exists)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the entire folder
api.upload_folder(
    folder_path=local_folder_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="upload inference results on 500k-sample test set using checkpoint 200"
)

print(f"✅ Upload complete! View at: https://huggingface.co/{repo_id}")

✅ Upload complete! View at: https://huggingface.co/dzungpham/SLA-SemEval-challenge
